In [1]:
import torch
import optuna
import pandas as pd
import yaml
from pathlib import Path
import json
import sys
sys.path.append('../')

from src.training.hparam_search import objective
from src.training.trainer import train_model, evaluate
from src.data.datasets import build_dataloaders
from src.models.lstm_classifier import LSTMClassifier

d:\Dataset\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

try:
    with open('../config/params.yml', 'r') as f:
        params = yaml.safe_load(f)
except FileNotFoundError:
    print("\nERROR: config/params.yml no encontrado.")
    params = None
except Exception as e:
    print(f"\nERROR al leer config/params.yml: {e}")
    params = None

if not params:
    print("\nNo se pudo cargar la configuración.")
    sys.exit(1)

seed = params.get('random_seed', 42)
exp_params = params.get('experiment', {})
exp_name = exp_params.get('name', 'default_experiment')
exp_base_dir = Path(exp_params.get('base_dir', '../experiments'))
experiment_dir = exp_base_dir / exp_name

print(f"\nCargando configuración para el experimento: {exp_name}")
print(f"Directorio del experimento: {experiment_dir.resolve()}")

dp_params = params.get('data_pipeline', {})
input_clips_dir = experiment_dir / dp_params.get('splitted_clips_output_subdir', ' processed_data/clips_splitted')

fe_params = params.get('feature_extraction', {})
training_params = params.get('training', {})
optuna_params = params.get('optuna_search', {})
model_params = params.get('lstm_model', {})
artifact_params = params.get('model_artifacts', {})

features_root = experiment_dir / fe_params.get('features_output_subdir', 'processed_data/features')
fe_extractor = fe_params.get('extractor', 'r3d')

manifests_subdir = experiment_dir / dp_params.get('manifests_subdir', 'results/tables')
models_subdir = experiment_dir / artifact_params.get('models_subdir', 'results/models')

features_root.mkdir(parents=True, exist_ok=True)
manifests_subdir.mkdir(parents=True, exist_ok=True)
models_subdir.mkdir(parents=True, exist_ok=True)

optuna_results_csv = manifests_subdir / optuna_params.get('results_csv_name', 'optuna_lstm_trials.csv')
final_model_path = models_subdir / artifact_params.get('final_model_name', 'best_lstm_model.pth')
final_metrics_path = manifests_subdir / artifact_params.get('final_metrics_name', 'lstm_final_metrics.json')
training_history_path = manifests_subdir / artifact_params.get('training_history_name', 'lstm_training_history.json')
test_predictions_path = manifests_subdir / artifact_params.get('test_predictions_name', 'lstm_test_predictions.json')

print(f"\nDirectorio de clips de entrada: {input_clips_dir.resolve()}")
print(f"Directorio de features de salida: {features_root.resolve()}")
print(f"Path del modelo final: {final_model_path.resolve()}")
print(f"Path de métricas finales: {final_metrics_path.resolve()}")
print(f"Path del historial de entrenamiento: {training_history_path.resolve()}")


Usando dispositivo: cuda

Cargando configuración para el experimento: exp_13
Directorio del experimento: D:\Dataset\experiments\exp_13

Directorio de clips de entrada: D:\Dataset\experiments\exp_13\processed_data\clips_splitted
Directorio de features de salida: D:\Dataset\experiments\exp_13\processed_data\features
Path del modelo final: D:\Dataset\experiments\exp_13\results\models\best_lstm_model.pth
Path de métricas finales: D:\Dataset\experiments\exp_13\results\tables\lstm_final_metrics.json
Path del historial de entrenamiento: D:\Dataset\experiments\exp_13\results\tables\lstm_training_history.json


## 1. Extracción de features 

Se selecciona el extractor en la llamada al script

### Extracción con R3D-18

In [ ]:
%run ../src/features/build_features.py \
--input_dir {input_clips_dir} \
--output_dir {features_root} \
--extractor {fe_extractor}


Usando dispositivo: cuda
Cargando modelo R3D_18
Procesando: experiments\exp_13\processed_data\clips_splitted\test\Normal


test/Normal: 100%|██████████| 39/39 [01:10<00:00,  1.82s/it]


Procesando: experiments\exp_13\processed_data\clips_splitted\test\Robbery


test/Robbery: 100%|██████████| 25/25 [00:43<00:00,  1.73s/it]


Procesando: experiments\exp_13\processed_data\clips_splitted\train\Normal


...

train/Robbery: 100%|██████████| 81/81 [02:18<00:00,  1.71s/it]


Procesando: experiments\exp_13\processed_data\clips_splitted\val\Normal


val/Normal: 100%|██████████| 31/31 [00:52<00:00,  1.70s/it]


Procesando: experiments\exp_13\processed_data\clips_splitted\val\Robbery


val/Robbery: 100%|██████████| 20/20 [00:34<00:00,  1.75s/it]


# 2. Entrenamiento de la LSTM

In [4]:
print(f"Búsqueda de hiperparámetros con Optuna")
sampler = optuna.samplers.TPESampler(seed=seed)
study = optuna.create_study(
    direction="minimize", 
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10, interval_steps=1),
    sampler=sampler
)

study.optimize(
    lambda trial: objective(
        trial,
        features_root=features_root,
        device=device,
        num_epochs=training_params["epochs"],
        patience=training_params["patience"],
        input_size=model_params.get("input_size", 512),
        seed=seed
    ),
    n_trials=optuna_params["n_trials"],
)

print("\n Búsqueda terminada")
print(f"Mejor trial Loss (validación): {study.best_trial.value:.4f}")
print("Mejores hiperparámetros:")
best_params = study.best_trial.params
print(best_params)

trials_df = study.trials_dataframe()
trials_df.to_csv(optuna_results_csv, index=False)

print(f"\n Resultados de todos los trials guardados en: {optuna_results_csv}")

[I 2025-12-06 14:44:47,407] A new study created in memory with name: no-name-cdc7abcb-6147-4106-a9c0-904cefcbbd13


Búsqueda de hiperparámetros con Optuna


[I 2025-12-06 14:45:01,880] Trial 0 finished with value: 0.5642409324645996 and parameters: {'hidden_size': 96, 'num_layers': 1, 'bidirectional': False, 'lr': 0.0011504753106625046, 'weight_decay': 0.0002607024758370766, 'batch_size': 32, 'dropout_fc': 0.28493564427131046}. Best is trial 0 with value: 0.5642409324645996.
[I 2025-12-06 14:45:10,002] Trial 1 finished with value: 0.6278167068958282 and parameters: {'hidden_size': 192, 'num_layers': 1, 'bidirectional': True, 'lr': 0.0007496501097315302, 'weight_decay': 5.4041038546473305e-05, 'batch_size': 32, 'dropout_fc': 0.4056937753654446}. Best is trial 0 with value: 0.5642409324645996.
[I 2025-12-06 14:45:19,644] Trial 2 finished with value: 0.6266907788813114 and parameters: {'hidden_size': 128, 'num_layers': 3, 'bidirectional': True, 'lr': 0.0007627211116577326, 'weight_decay': 1.5679933916722995e-05, 'batch_size': 16, 'dropout_fc': 0.39807076404450803}. Best is trial 0 with value: 0.5642409324645996.
[I 2025-12-06 14:45:28,568] Tr


 Búsqueda terminada
Mejor trial Loss (validación): 0.4487
Mejores hiperparámetros:
{'hidden_size': 64, 'num_layers': 2, 'bidirectional': True, 'lr': 0.000820393954491359, 'weight_decay': 3.433623307618017e-05, 'batch_size': 64, 'dropout_fc': 0.323819408991232}

 Resultados de todos los trials guardados en: ..\experiments\exp_13\results\tables\optuna_lstm_trials.csv


In [5]:
print(f"Entrenando modelo final con los mejores hiperparámetros")

final_config = best_params.copy()
final_config["input_size"] = model_params.get("input_size", 512)

history, best_val_loss, best_val_auc, best_epoch, test_metrics, test_predictions = train_model(
    config=final_config,
    features_root=features_root,
    save_path=final_model_path,
    device=device,
    num_epochs=training_params["epochs"],
    patience=training_params["patience"],
    seed=seed
)

final_results_summary = {
    "best_hyperparameters": best_params,
    "best_validation_loss": best_val_loss,
    "best_validation_auc": best_val_auc,
    "best_epoch": best_epoch,
    "test_metrics": test_metrics,
    "features_used": str(features_root)
}

with open(final_metrics_path, "w") as f:
    json.dump(final_results_summary, f, indent=4)

with open(training_history_path, "w") as f:
    json.dump(history, f, indent=4)

with open(test_predictions_path, "w") as f:
    json.dump(test_predictions, f, indent=4)

print(f"\nMétricas finales del test guardadas en: {final_metrics_path.resolve()}")
print(f"Modelo final entrenado guardado en: {final_model_path.resolve()}")
print(f"Historial de entrenamiento guardado en: {training_history_path.resolve()}")
print(f"Predicciones del test guardadas en: {test_predictions_path.resolve()}")

Entrenando modelo final con los mejores hiperparámetros
Entrenando con Config: {'hidden_size': 64, 'num_layers': 2, 'bidirectional': True, 'lr': 0.000820393954491359, 'weight_decay': 3.433623307618017e-05, 'batch_size': 64, 'dropout_fc': 0.323819408991232, 'input_size': 512}

Epoch [1/60] | Train Loss: 0.6749 | Val Loss: 0.6639 | Val AUC: 0.7516
 Mejor modelo guardado en epoch 1 con Val AUC: 0.7516. Guardado en D:\Dataset\experiments\exp_13\results\models\best_lstm_model.pth
Epoch [2/60] | Train Loss: 0.6559 | Val Loss: 0.6468 | Val AUC: 0.7935
 Mejor modelo guardado en epoch 2 con Val AUC: 0.7935. Guardado en D:\Dataset\experiments\exp_13\results\models\best_lstm_model.pth
Epoch [3/60] | Train Loss: 0.6231 | Val Loss: 0.6184 | Val AUC: 0.7871
 Mejor modelo guardado en epoch 3 con Val AUC: 0.7871. Guardado en D:\Dataset\experiments\exp_13\results\models\best_lstm_model.pth
Epoch [4/60] | Train Loss: 0.5868 | Val Loss: 0.5624 | Val AUC: 0.8242
 Mejor modelo guardado en epoch 4 con Val A